# 3DGS Pipeline 控制中枢

**使用方法**：
1. 运行 **Cell 1（初始化）**，加载所有函数和配置
2. 按需运行各 Section 中的单元格
3. 修改参数？编辑 ，重新运行 Cell 1 即可

> 每个 Section 均可独立运行，无需依赖上方单元格的执行状态。

In [56]:
# ═══════════════════════════════════════════════════════
# ① 初始化（每次打开 Notebook 只需运行这一个 Cell）
# ═══════════════════════════════════════════════════════
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

os.environ["CUDA_DEVICE_ORDER"]        = "PCI_BUS_ID"          # 让 CUDA 编号与 nvidia-smi 一致
os.environ["CUDA_VISIBLE_DEVICES"]     = "0"                    # 0 = RTX 2080 Ti
os.environ["PYTORCH_CUDA_ALLOC_CONF"]  = "expandable_segments:False"

# 确保 src/ 在 Python 路径中（支持从任意目录打开 notebook）
_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / "src" / "pipeline" / "__init__.py").exists()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.pipeline import *

cfg = load_config()   # 读取 configs/pipeline.yaml
print(f"✓ 项目根目录 : {PROJECT_ROOT}")
print(f"✓ 数据集     : {cfg['dataset']['source']}  →  {cfg['dataset']['path']}")
print(f"✓ 训练输出   : {cfg['training']['output_dir']}")
print(f"✓ 迭代次数   : {cfg['training']['iterations']}")
print(f"✓ 查看器     : {cfg['viewer']['backend']}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ 项目根目录 : /home/ansatz/github/ME6402-3D-Autonomous-Retail
✓ 数据集     : custom  →  /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/team_data2
✓ 训练输出   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter
✓ 迭代次数   : 30000
✓ 查看器     : sibr


## Section 1 — 环境检查

In [38]:
# 检查 PyTorch、CUDA、COLMAP、3DGS CUDA 模块、Open3D、Docker
check_environment(cfg)


2026-04-05 19:30:28,090  INFO  环境检查完成，结果: 通过


3DGS 环境检查

PyTorch:  2.1.2
CUDA 可用: True
CUDA 版本: 11.8
  GPU 0: NVIDIA GeForce RTX 2080 Ti  (21.7 GB)

核心依赖:
  ✓ OpenCV  4.8.1
  ✓ NumPy  1.26.4
  ✓ plyfile
  ✓ SciPy  1.11.4
  ✓ Open3D  0.19.0
  ✓ diff_gaussian_rasterization (CUDA 模块)

COLMAP:
  ✓ /usr/bin/colmap

Docker（SIBR 查看器）:
  ✓ Docker daemon 可访问

项目目录:
  PROJECT_ROOT : /home/ansatz/github/ME6402-3D-Autonomous-Retail
  GS_DIR       : /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/gaussian-splatting  ✓
  DATA_DIR     : /home/ansatz/github/ME6402-3D-Autonomous-Retail/data  ✓
  OUTPUT_DIR   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs  ✓

✅ 环境检查通过


True

## Section 2 — 视频抽帧（可选）

适用场景：你有一段录制的视频，需要先抽帧再走 COLMAP 流程。

**配置方式**：在  中修改  块，将 ，
然后重新运行 **Cell 1**，再运行本 Cell。

In [19]:
# 抽帧完成后会自动更新 cfg，指向新场景目录
# 完成后直接运行 Section 3 (COLMAP) 即可
extract_frames(cfg)


ℹ️  video.enabled=false，跳过抽帧。
   若要抽帧，请在 configs/pipeline.yaml 中将 video.enabled 改为 true。


False

## Section 3 — COLMAP 相机标定

适用场景：自有数据（视频抽帧或自拍照片），需要从图像推导相机参数。
官方数据集（T&T、DB）已自带相机参数，**无需此步骤**。

**配置方式**：在  中将 ，
并确认  和  正确。

In [20]:
# COLMAP 五步流程：特征提取 → 匹配 → 稀疏重建 → 畸变校正 → 内参回填
# 完成后 cfg["dataset"]["path"] 自动切换到 undistorted dense 输出
run_colmap(cfg)



🧭 运行 COLMAP  →  /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2
   bash /home/ansatz/github/ME6402-3D-Autonomous-Retail/scripts/reconstruction/run_colmap.sh /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/team_data2/images /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2

Feature extraction

Processed file [1/117]
  Name:            IMG_20260402_120500.jpg
  SKIP: Features for image already extracted.
Processed file [2/117]
  Name:            IMG_20260402_120505.jpg
  SKIP: Features for image already extracted.
Processed file [3/117]
  Name:            IMG_20260402_120508.jpg
  SKIP: Features for image already extracted.
Processed file [4/117]
  Name:            IMG_20260402_120512.jpg
  SKIP: Features for image already extracted.
Processed file [5/117]
  Name:            IMG_20260402_120516.jpg
  SKIP: Features for image already extracted.
Processed file [6/117]
  Name:            IMG_20260402_120523.jpg
  SKIP: Fe

KeyboardInterrupt: 

## Section 4 — 3DGS 训练

关键参数（在  →  块修改）：
- ：迭代次数（300~5000 快速验证；30000 高质量）
- ：分辨率倍率（1=原始；2=1/2；RTX 4060 建议从 2 开始）
- ：输出根目录（每次训练自动创建子目录）

In [ ]:
# OOM 时自动降档重试（resolution ×1 → ×2 → ×4）
run_training(cfg)


2026-04-05 00:58:31,120  INFO  ==================================================
2026-04-05 00:58:31,120  INFO  启动 3DGS 训练
2026-04-05 00:58:31,121  INFO  ==================================================
2026-04-05 00:58:31,121  INFO  训练前自动修正 dataset.path: /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense
2026-04-05 00:58:31,121  INFO  训练命令: python train.py -s /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense -m /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter --iterations 30000 --resolution 2 --sh_degree 3 --save_iterations 7000 30000 --test_iterations 7000 30000


🧭 训练前自动修正数据路径 →  /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense

⏳ 训练开始
   数据   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense
   迭代   : 30000
   输出   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter
   分辨率 : ×2
Optimizing /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter
Output folder: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter [05/04 00:58:31]
Tensorboard not available: not logging progress [05/04 00:58:31]

Reading camera 1/117
Reading camera 2/117
Reading camera 3/117
Reading camera 4/117
Reading camera 5/117
Reading camera 6/117
Reading camera 7/117
Reading camera 8/117
Reading camera 9/117
Reading camera 10/117
Reading camera 11/117
Reading camera 12/117
Reading camera 13/117
Reading camera 14/117
Reading camera 15/117
Reading camera 16/117
Reading camera 17/117
Reading camera 18/117
Reading camera 19/117
Read

2026-04-05 01:47:04,903  INFO  训练完成，耗时 48.6min，输出=/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter



✅ 训练完成  （48.6 分钟）
   模型输出: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter


True

## Section 5 — 查看训练结果

自动搜索  下最新的 ，用 Open3D 打开交互窗口。

如需查看 SIBR，在  中将 ，
重新运行 Cell 1 后再运行此 Section。

In [ ]:
# Open3D 交互查看（关闭窗口后继续）
# 也可传入指定路径：open_viewer(cfg, ply_path="outputs/xxx/point_cloud/iteration_300/point_cloud.ply")
open_viewer(cfg)


✅ 找到点云：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter/point_cloud/iteration_7000/point_cloud.ply
   大小：75.7 MB
   点数：320,263
   打开 Open3D 交互窗口（关闭窗口后继续）...


2026-04-05 01:47:38,939  INFO  Open3D 查看完成: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter/point_cloud/iteration_7000/point_cloud.ply


True

In [ ]:
# 分析训练结果：列出 PLY 文件、打印 results.json
analyze_results(cfg)


## Section 6 — SIBR Viewer（可选）

如需在 SIBR 中查看，运行此 Cell。会列出可用模型供你选择。

**前提**：已构建 Docker 镜像（见 [info] Building image: sibr-builder:ubuntu22.04-cuda11.8
Sending build context to Docker daemon  12.01GB

Step 1/4 : FROM nvidia/cuda:11.8.0-devel-ubuntu22.04
 ---> 6f9cc9f1ba9e
Step 2/4 : ENV DEBIAN_FRONTEND=noninteractive
 ---> Using cache
 ---> db877d8112e1
Step 3/4 : RUN apt-get update && apt-get install -y --no-install-recommends     build-essential     cmake     ninja-build     git     pkg-config     libglew-dev     libassimp-dev     libboost-all-dev     libgtk-3-dev     libopencv-dev     libglfw3-dev     libavdevice-dev     libavcodec-dev     libavformat-dev     libswscale-dev     libeigen3-dev     libxxf86vm-dev     libembree-dev     libgl1-mesa-dev     libglu1-mesa-dev     libx11-dev     libxext-dev     libxrender-dev     libxrandr-dev     libxinerama-dev     libxcursor-dev     ca-certificates     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> feb68a7c28aa
Step 4/4 : WORKDIR /workspace
 ---> Using cache
 ---> 8fdbd6c2ecd5
Successfully built 8fdbd6c2ecd5
Successfully tagged sibr-builder:ubuntu22.04-cuda11.8
[done] Image built: sibr-builder:ubuntu22.04-cuda11.8）

In [ ]:
# 交互式选择模型并在 Docker 内启动 SIBR
launch_sibr(cfg)



可用模型（按最新迭代降序）:
  [1] team_data2_30000iter  (iter=30000)
  [2] 7dfdb283-b  (iter=30000)
  [3] 3dgs_team_data2_30000iter  (iter=30000)
  [4] 3dgs_tandt_30000iter  (iter=30000)
  [5] 3dgs_tandt_5000iter  (iter=5000)
  [6] 3dgs_custom_scene_01_5000iter  (iter=5000)
  [7] 3dgs_demo_300iter  (iter=300)
  [8] smoke_truck_10iter  (iter=10)
  [9] 3dgs_demo  (iter=?)

🖼️  启动 SIBR Viewer ...
   bash /home/ansatz/github/ME6402-3D-Autonomous-Retail/scripts/reconstruction/run_sibr_in_docker.sh sibr-builder:ubuntu22.04-cuda11.8 /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   提示：关闭 SIBR 窗口后，该单元继续运行。
[info] Launching SIBR viewer in container...

== CUDA ==

CUDA Version 11.8.0

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://d

[SIBR] ##  ERROR  ##:	FILE /workspace/third_party/gaussian-splatting/SIBR_viewers/src/core/scene/ParseData.cpp
			LINE 560, FUNC getParsedData
			Cannot determine type of dataset at //root/gpufree-data/gaussian-splatting/image


[SIBR] --  INFOS  --:	Loading 435936 Gaussian splats
[SIBR] --  INFOS  --:	Initializing Raycaster
[SIBR] --  INFOS  --:	Interactive camera using (0.009,1100) near/far planes.
Switched to trackball mode.


2026-04-05 15:51:59,973  INFO  SIBR 查看完成: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter，ok=True


[SIBR] --  INFOS  --:	Deinitialization of GLFW
[done] SIBR viewer exited.
✓ SIBR 正常退出


True

## Section 7 — 一键完整流程

参数调好后，一口气执行：COLMAP（可选）→ 训练 → 查看结果。

In [ ]:
# 一键流程（是否跑 COLMAP 取决于 cfg["dataset"]["use_colmap"]）
run_pipeline(cfg)


## Section 9 — SAGA 3D 语义分割

**Segment Any 3D Gaussians**（AAAI 2025）：在已训练好的 3DGS 模型上直接添加语义特征，无需重新训练重建，无需标注数据。

**流程概览**
1. 安装 SAGA 依赖（只需做一次）
2. 下载 SAM ViT-H checkpoint（~2.5 GB，只需做一次）
3. 提取 SAM 特征 + mask（按场景做一次）
4. 训练对比特征（~10-40 分钟）
5. 打开 SAGA Notebook → 文字/点击 → 输出 3D Bounding Box

**配置方式**：在 `configs/pipeline.yaml` 的 `saga:` 块中设置 `image_root` 和 `model_path`，然后重新运行 **Cell 1**。

### 9.1  导入 SAGA 模块 + 加载配置


In [55]:
from src.pipeline.saga import *

saga_cfg = load_saga_config(cfg)   # 读取 pipeline.yaml 中的 saga: 块

print(f"模型路径  : {saga_cfg['model_path']}")
print(f"场景图像  : {saga_cfg['image_root']}")
print(f"SAM ckpt  : {saga_cfg['sam_checkpoint']}")
print(f"降采样    : ×{saga_cfg['downsample']}")


模型路径  : outputs/3dgs_team_data2_30000iter
场景图像  : data/colmap_workspace/team_data2/dense
SAM ckpt  : dependencies/sam_ckpt/sam_vit_h_4b8939.pth
降采样    : ×4


### 9.2  SAGA 环境检查（确认依赖全部就绪）

In [ ]:
check_saga_ready()


SAGA 环境检查
  ✓ SAGA 仓库  (/home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA)
  ✓ 子模块 diff-gaussian-rasterization_contrastive_f
  ✓ segment-anything (SAM)
  ✓ kmeans_pytorch
  ✓ open_clip_torch
  ✓ hdbscan
  ✓ SAM ViT-H checkpoint  (sam_vit_h_4b8939.pth)

✅ SAGA 环境就绪


True

In [54]:
# 9.3  首次使用：安装 SAGA 依赖
# 若 9.2 显示全部 ✓，可跳过本单元格
import subprocess, sys
result = subprocess.run(
    ["bash", "scripts/setup_saga.sh"],
    cwd=str(PROJECT_ROOT)
)
print("✅ 安装完成" if result.returncode == 0 else "✗ 安装失败，查看上方输出")


 SAGA 安装脚本
 项目根目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail
✓ Conda 环境：gaussian_splatting
✓ SAGA 仓库已存在，更新子模块...

>>> 安装 segment-anything...
✓ segment-anything 安装完成（本地源）

>>> 安装 kmeans_pytorch...
✓ kmeans_pytorch 安装完成（本地源）

>>> 安装 open_clip_torch, hdbscan...
✓ open_clip_torch, hdbscan 安装完成

>>> 编译 diff-gaussian-rasterization_contrastive_f...


ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# 9.4  下载 SAM ViT-H checkpoint（~2.5 GB，只需一次）
# 若 dependencies/sam_ckpt/sam_vit_h_4b8939.pth 已存在可跳过
download_sam_checkpoint()


✓ SAM checkpoint 已存在：/home/ansatz/github/ME6402-3D-Autonomous-Retail/dependencies/sam_ckpt/sam_vit_h_4b8939.pth


True

### 9.5 — 指定要分割的场景和模型

修改下方两个变量（或直接在 `configs/pipeline.yaml` 的 `saga:` 块中修改后重新运行 Cell 1）：

```
saga:
  image_root: data/official/tandt_db/db/playroom   # 场景图像目录（含 images/ 子目录）
  model_path: outputs/playroom_30000iter            # 训练好的 3DGS 模型目录
```

> ⚠️ Teammate 的模型（`Teamate_output_unzipped/...`）训练于 Google Colab，本地没有对应图像，需使用本地训练的模型或用自己拍摄的数据。

In [ ]:
# 9.5  确认场景和模型路径（由 configs/pipeline.yaml 的 saga: 块控制）
# 如需临时覆盖，取消下方注释并修改：
# saga_cfg["image_root"] = str(PROJECT_ROOT / "data/colmap_workspace/custom_scene_01/dense")
# saga_cfg["model_path"] = str(PROJECT_ROOT / "outputs/3dgs_custom_scene_01_5000iter")

print(f"模型路径  : {saga_cfg['model_path']}")
print(f"场景图像  : {saga_cfg['image_root']}")
print(f"SAM ckpt  : {saga_cfg['sam_checkpoint']}")
print(f"降采样    : ×{saga_cfg['downsample']}")


模型路径  : outputs/3dgs_team_data2_30000iter
场景图像  : data/colmap_workspace/team_data2/dense
SAM ckpt  : dependencies/sam_ckpt/sam_vit_h_4b8939.pth
降采样    : ×1


### 9.6 — 提取特征 + 训练（按顺序 9.6.1 → 9.6.2 → 9.6.3 → 9.6.4 运行一次即可）

每个场景只需运行一次，结果会缓存在场景目录下。

| 步骤 | 单元格 | 输出目录 | 耗时 |
|------|--------|----------|------|
| SAM mask 提取 | 9.6.1 | `sam_masks/` | ~6 分钟 |
| CLIP 特征提取 | 9.6.2 | `clip_features/` | ~2 分钟 |
| 3D 尺度估算 | 9.6.3 | `mask_scales/` | ~3 分钟 |
| 对比特征训练 | 9.6.4 | `point_cloud/.../contrastive_*.ply` | 10-40 分钟 |


#### 9.6.1  Step 1：生成缩小图像 + 提取 SAM 自动分割 mask（输出到 <image_root>/sam_masks/）

In [ ]:
# VRAM：~7 GB，建议在 RTX 2080 Ti 上运行
extract_sam_masks(saga_cfg)


✓ images_4/ 已存在，跳过创建。

[SAGA 9.6.1] 提取 SAM 自动分割 mask...
   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/images_4
   输出目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/sam_masks
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/extract_segment_everything_masks.py --image_root ...
Initializing SAM...
Extracting SAM segment everything masks...

100%|██████████| 117/117 [06:24<00:00,  3.29s/it]
✓ 完成


True

#### 9.6.2  Step 2：从图像 + SAM mask 提取 CLIP 语义特征（输出到 <image_root>/clip_features/）


In [ ]:
# VRAM：~4 GB
extract_sam_features(saga_cfg)


   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/images（117 张）

[SAGA 9.6.2] 提取 CLIP 语义特征（从 SAM mask）...
   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/images
   输出目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/clip_features
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/get_clip_features.py --image_root ...
Embedding dimension 512

0it [00:00, ?it/s]/home/ansatz/miniconda3/envs/gaussian_splatting/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future de

True

#### 9.6.3  Step 3：估算 mask 3D 物理尺度（输出到 <image_root>/mask_scales/）


In [ ]:
# 训练前必须运行，约 3-5 分钟
compute_scales(saga_cfg)



[SAGA 9.6.3] 估算 mask 3D 物理尺度...
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/get_scale.py -m ...
Looking for config file in /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Config file found: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Loading trained model at iteration 30000, None
Allow Camera Principle Point Shift: False

Reading camera 1/117
Reading camera 2/117
Reading camera 3/117
Reading camera 4/117
Reading camera 5/117
Reading camera 6/117
Reading camera 7/117
Reading camera 8/117
Reading camera 9/117
Reading camera 10/117
Reading camera 11/117
Reading camera 12/117
Reading camera 13/117
Reading camera 14/117
Reading camera 15/117
Reading camera 16/117
Reading camera 17/117
Reading camera 18/117
Reading camera 19/117
Reading camera 20/117
Reading camera 21/117
Reading camera 22/117
Reading camera 23/117
Re

True

#### 9.6.4  Step 4：在冻结的 3DGS 上训练对比特征（~10-40 分钟）


In [ ]:
# 输出：<model_path>/point_cloud/.../contrastive_feature_point_cloud.ply
train_saga_features(saga_cfg)



[SAGA 9.6.4] 训练对比特征...
   模型路径：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   场景数据：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense
   预计时间：10~40 分钟
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/train_contrastive_feature.py -m ...
Looking for config file in /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Config file found: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Optimizing /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
RFN weight: 1.0 [05/04 18:20:02]
Smooth K: 16 [05/04 18:20:02]
Scale aware dim: -1 [05/04 18:20:02]
Loading trained model at iteration 30000, None [05/04 18:20:02]
Allow Camera Principle Point Shift: True [05/04 18:20:02]

Reading camera 1/117
Reading camera 2/117
Reading camera 3/117
R

True

### 9.7 SAGA 交互GUI分割和特征识别

#### 9.7.1  方式 A：启动 SAGA 交互 GUI（点击分割，推荐）

In [ ]:
# 操作：勾选 clickmode → 右键点击物体 → segment3d → save as
open_saga_gui(saga_cfg, gpu=0)



[SAGA GUI] 启动交互式分割界面...
   模型：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   场景 iter：30000，特征 iter：10000，GPU：0
   操作：勾选 clickmode → 右键点击物体 → segment3d → save as
✓ GUI 已启动，等待窗口弹出（约 10-20 秒）


loading model file...
project mat initialized !
loading model file done.
clickmode_multi_button =  True
[788.0, 240.0]
[775.0, 268.0]
[764.0, 290.0]
Saving ...
Clustering in 3D...
Clustering finished.
[761.0, 292.0]
[975.0, 273.0]
[453.0, 458.0]
[679.0, 344.0]
[671.0, 402.0]
[747.0, 270.0]
[407.0, 184.0]
[570.0, 243.0]
[479.0, 251.0]
[399.0, 261.0]
[462.0, 301.0]
[532.0, 327.0]
[612.0, 269.0]
[604.0, 340.0]
[640.0, 247.0]
[668.0, 331.0]
[681.0, 250.0]
[697.0, 64.0]
[633.0, 145.0]
[509.0, 310.0]
[587.0, 391.0]
[723.0, 209.0]
[660.0, 299.0]
[554.0, 234.0]
[734.0, 311.0]
[745.0, 239.0]
[725.0, 384.0]
[908.0, 326.0]
[845.0, 339.0]
[699.0, 451.0]
[824.0, 518.0]
[722.0, 293.0]
[752.0, 288.0]
[772.0, 292.0]
[726.0, 292.0]
[749.0, 246.0]
[741.0, 281.0]


#### 9.7.2  文字查询 → 自动定位物体 → 3D Bounding Box


In [57]:
result = query_by_text("coca cola can", saga_cfg)
print(result)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 11995
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-05 20:09:45,887  INFO  query_by_text [coca cola can]: center=[-7.340358734130859, 0.2434229850769043, 5.107248306274414], size=[2.991562604904175, 2.368671417236328, 1.605008602142334]
INFO:pipeline:query_by_text [coca cola can]: center=[-7.340358734130859, 0.2434229850769043, 5.107248306274414], size=[2.991562604904175, 2.368671417236328, 1.605008602142334]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/coca_cola_can.pt  (选中 Gaussian 数：449)

✅ 3D Bounding Box — coca cola can
   中心坐标 : [-7.3404, 0.2434, 5.1072]  （单位：米）
   尺寸 XYZ : [2.9916, 2.3687, 1.6050]
   包含 Gaussian 数 : 449
{'label': 'coca cola can', 'center': [-7.340358734130859, 0.2434229850769043, 5.107248306274414], 'size': [2.991562604904175, 2.368671417236328, 1.605008602142334], 'bbox_min': [-8.836139678955078, -0.9409127235412598, 4.304743766784668], 'bbox_max': [-5.844577312469482, 1.4277586936950684, 5.90975284576416], 'n_gaussians': 449, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/coca_cola_can.pt'}


In [59]:
result2 = query_by_text("facial tissue box", saga_cfg, score_threshold=0.7)
print(result2)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 12207
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-05 20:12:56,647  INFO  query_by_text [facial tissue box]: center=[-0.7388391494750977, 1.7515795230865479, -0.9173855781555176], size=[19.855316162109375, 13.53880500793457, 29.368711471557617]
INFO:pipeline:query_by_text [facial tissue box]: center=[-0.7388391494750977, 1.7515795230865479, -0.9173855781555176], size=[19.855316162109375, 13.53880500793457, 29.368711471557617]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/facial_tissue_box.pt  (选中 Gaussian 数：1373)

✅ 3D Bounding Box — facial tissue box
   中心坐标 : [-0.7388, 1.7516, -0.9174]  （单位：米）
   尺寸 XYZ : [19.8553, 13.5388, 29.3687]
   包含 Gaussian 数 : 1,373
{'label': 'facial tissue box', 'center': [-0.7388391494750977, 1.7515795230865479, -0.9173855781555176], 'size': [19.855316162109375, 13.53880500793457, 29.368711471557617], 'bbox_min': [-10.666497230529785, -5.017823219299316, -15.601741790771484], 'bbox_max': [9.18881893157959, 8.520981788635254, 13.766969680786133], 'n_gaussians': 1373, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/facial_tissue_box.pt'}


In [60]:
result3 = query_by_text("potato chips bag", saga_cfg)
print(result3)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 12030
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-05 20:14:07,720  INFO  query_by_text [potato chips bag]: center=[-1.113645315170288, -2.0086467266082764, 11.485057830810547], size=[15.952309608459473, 14.880890846252441, 23.319904327392578]
INFO:pipeline:query_by_text [potato chips bag]: center=[-1.113645315170288, -2.0086467266082764, 11.485057830810547], size=[15.952309608459473, 14.880890846252441, 23.319904327392578]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/potato_chips_bag.pt  (选中 Gaussian 数：1504)

✅ 3D Bounding Box — potato chips bag
   中心坐标 : [-1.1136, -2.0086, 11.4851]  （单位：米）
   尺寸 XYZ : [15.9523, 14.8809, 23.3199]
   包含 Gaussian 数 : 1,504
{'label': 'potato chips bag', 'center': [-1.113645315170288, -2.0086467266082764, 11.485057830810547], 'size': [15.952309608459473, 14.880890846252441, 23.319904327392578], 'bbox_min': [-9.089799880981445, -9.449091911315918, -0.1748943328857422], 'bbox_max': [6.862509727478027, 5.431798934936523, 23.145009994506836], 'n_gaussians': 1504, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/potato_chips_bag.pt'}


In [61]:
result4 = query_by_text("juice drink pouch", saga_cfg)
print(result4)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 11869
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-05 20:15:12,202  INFO  query_by_text [juice drink pouch]: center=[-6.992176055908203, 1.2619150876998901, 4.845025062561035], size=[4.349524021148682, 3.8626885414123535, 2.7883310317993164]
INFO:pipeline:query_by_text [juice drink pouch]: center=[-6.992176055908203, 1.2619150876998901, 4.845025062561035], size=[4.349524021148682, 3.8626885414123535, 2.7883310317993164]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/juice_drink_pouch.pt  (选中 Gaussian 数：907)

✅ 3D Bounding Box — juice drink pouch
   中心坐标 : [-6.9922, 1.2619, 4.8450]  （单位：米）
   尺寸 XYZ : [4.3495, 3.8627, 2.7883]
   包含 Gaussian 数 : 907
{'label': 'juice drink pouch', 'center': [-6.992176055908203, 1.2619150876998901, 4.845025062561035], 'size': [4.349524021148682, 3.8626885414123535, 2.7883310317993164], 'bbox_min': [-9.166937828063965, -0.6694291830062866, 3.450859546661377], 'bbox_max': [-4.817414283752441, 3.1932592391967773, 6.239190578460693], 'n_gaussians': 907, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/juice_drink_pouch.pt'}


#### 9.7.3  渲染分割结果（render.py）

将分割出的物体渲染成图片，查看每个训练视角下的效果：
- `--target scene --segment`：渲染带分割的场景（背景移除）
- `--target seg`：渲染 2D 黑白 mask 图

In [ ]:
import subprocess, os
from pathlib import Path

model_path = PROJECT_ROOT / saga_cfg["model_path"]
mask_path  = model_path / "segmentation_res" / "mouse.pt"

result = subprocess.run(
    [
        "python", "render.py",
        "-m", str(model_path),
        "--precomputed_mask", str(mask_path),
        "--target", "scene",
        "--segment",
        "--allow_principle_point_shift",
    ],
    cwd=str(PROJECT_ROOT / "third_party/SAGA"),
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "1"},
)

if result.returncode == 0:
    out_dir = model_path / "train" / f"ours_{saga_cfg.get('scene_iteration', 5000)}"
    print(f"✅ 渲染完成，图片输出到：{out_dir}")
else:
    print("✗ 渲染失败，查看上方输出")

Looking for config file in /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_custom_scene_01_5000iter/cfg_args
Config file found: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_custom_scene_01_5000iter/cfg_args
Rendering /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_custom_scene_01_5000iter
Using precomputed mask /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_custom_scene_01_5000iter/segmentation_res/mouse.pt
Loading trained model at iteration 5000, None [04/04 23:28:32]
Allow Camera Principle Point Shift: True [04/04 23:28:32]
Reading camera 198/198 [04/04 23:28:32]
Loading Training Cameras [04/04 23:28:32]
Loading Test Cameras [04/04 23:28:39]


Rendering progress: 100%|██████████| 198/198 [00:28<00:00,  6.92it/s]
Rendering progress: 0it [00:00, ?it/s]


✅ 渲染完成，图片输出到：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_custom_scene_01_5000iter/train/ours_5000


#### 9.7.4  3D 识别框查看器（高斯泼溅真实感渲染 + YOLO 风格 3D 框）

独立查看器，与 SAGA GUI 互不干扰：
- **高斯泼溅真实感渲染**（和 SAGA GUI 同级画质）
- **黄色 3D 线框 + 标签**，旋转视角时框随透视变换
- 左键旋转 / 右键平移 / 滚轮缩放
- 支持同时显示多个物体识别框（传入 list）


In [ ]:
# 使用上一步 query_by_text 返回的 bbox 直接启动查看器
# 也可传入多个：open_bbox_viewer([bbox1, bbox2], saga_cfg, gpu=1)

bboxes = [result]  # result 是之前 coca cola can 的结果
open_bbox_viewer(bboxes, saga_cfg)




[BBox Viewer] 启动 3D 识别框查看器...
   模型：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   识别物体：['coca cola can']，GPU：1
   操作：左键旋转 / 右键平移 / 滚轮缩放
✓ 查看器已启动，等待窗口弹出（约 10-20 秒）


[BBoxViewer] Loading model: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
[BBoxViewer] PLY: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/point_cloud/iteration_30000/point_cloud.ply
[BBoxViewer] Loaded 1,201,772 Gaussians
